# 02 · Adaptation, preferences and rewards

Mathematics without starting training. The runnable optional recipes live in `scripts/train.py` and `scripts/preferences.py`. This notebook does not estimate model quality.

## LoRA

$W^{\prime}=W+\frac{\alpha}{r}BA$, with $A\in\mathbb{R}^{r\times d_{in}}$ and $B\in\mathbb{R}^{d_{out}\times r}$. Freeze $W$ and optimize $A,B$. The adapter parameter count is $r(d_{in}+d_{out})$ rather than $d_{in}d_{out}$.

In [ ]:
din,dout,r=2048,2048,16
full=din*dout
lora=r*(din+dout)
print({"full_matrix":full,"adapter":lora,"fraction":lora/full})
assert lora < full

## QLoRA memory accounting

A 4-bit backbone stores quantized weights, scales and metadata. Gradients flow through dequantized operations into adapters. Optimizer state and activations still consume memory; multiplying parameter count by 0.5 bytes is **not** total training memory. NF4 and double quantization follow Dettmers et al. (2023), https://arxiv.org/abs/2305.14314.

In [ ]:
parameters=800_000_000
for bits in [32,16,4]:
    print(bits, "bit weight-only lower bound:", round(parameters*bits/8/2**30,3), "GiB")

## DPO

$L_{DPO}=-\log\sigma[\beta((\log\pi_\theta(y_w|x)-\log\pi_{ref}(y_w|x))-(\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_l|x)))]$

A chosen response should gain relative log probability against the rejected one, constrained by the reference. Human preferences and factual correctness are not interchangeable. Source: https://arxiv.org/abs/2305.18290.

In [ ]:
import numpy as np
def dpo_loss(policy_chosen,policy_rejected,ref_chosen,ref_rejected,beta=.1):
    margin=beta*((policy_chosen-ref_chosen)-(policy_rejected-ref_rejected))
    return float(np.logaddexp(0,-margin))
assert dpo_loss(-1,-3,-2,-2) < dpo_loss(-3,-1,-2,-2)
print(dpo_loss(-1,-3,-2,-2))

## GRPO group-relative advantage

$\hat A_i=(r_i-\bar r)/(s_r+\epsilon)$ normalizes rewards among completions for one prompt. This illustrates the relative baseline; a full GRPO objective also uses policy ratios, clipping and optional KL regularization. DeepSeekMath: https://arxiv.org/abs/2402.03300.

In [ ]:
rewards=np.array([0.,1.,1.,0.])
adv=(rewards-rewards.mean())/(rewards.std()+1e-8)
assert abs(adv.mean()) < 1e-6
print(adv)
print("Constant reward:", (np.ones(4)-1)/(1e-8))

## Research controls

Split by paper before tokenization. Audit long-context truncation and target support. Keep validation and original benchmark test questions out of preference/reward fitting. Compare base and adapter on both domain tasks and broad capabilities: a lower training loss can accompany worse general language or math performance.